# Sesión 04 — Pruebas Estadísticas y Validación de Modelos
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo I · Fundamentos del Aprendizaje Estadístico**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Visualizar y cuantificar la descomposición sesgo-varianza en modelos de distintas complejidades.
2. Elegir la estrategia de validación cruzada adecuada para datos fisiológicos, distinguiendo k-fold estándar de LOSO.
3. Demostrar cuantitativamente la fuga de información cuando el preprocesamiento se aplica antes de la CV.
4. Aplicar correcciones de Bonferroni y FDR (Benjamini-Hochberg) para comparaciones múltiples.
5. Implementar la prueba de permutaciones como alternativa no paramétrica a las pruebas paramétricas.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Varoquaux, G. & Poldrack, R.A. (2019). Predictive models avoid excessive optimism in clinical practice. *PNAS*, 116(6), 1861–1863. https://doi.org/10.1073/pnas.1816956116 |
| ★★★ | Kapoor, S. & Narayanan, A. (2023). Leakage and the reproducibility crisis in machine-learning-based science. *Patterns*, 4(9), 100804. https://doi.org/10.1016/j.patter.2023.100804 |
| ★★☆ | Benjamini, Y. & Hochberg, Y. (1995). Controlling the false discovery rate. *JRSS-B*, 57(1), 289–300. |
| ★★☆ | Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2ª ed.). Cap. 7. Springer. |
| ★☆☆ | Good, P. (2005). *Permutation, Parametric, and Bootstrap Tests of Hypotheses* (3ª ed.). Springer. |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Configuración completa.')

## Parte 1 — Descomposición sesgo-varianza

El error esperado de un modelo se descompone en tres términos:

$$\mathbb{E}[(y - \hat{f}(x))^2] = \underbrace{\text{Sesgo}^2[\hat{f}(x)]}_{\text{error sistemático}} + \underbrace{\text{Var}[\hat{f}(x)]}_{\text{sensibilidad a los datos}} + \underbrace{\sigma^2}_{\text{ruido irreducible}}$$

- **Sesgo alto** → modelo demasiado simple (subajuste / *underfitting*)
- **Varianza alta** → modelo demasiado complejo (sobreajuste / *overfitting*)
- **Ruido irreducible** → inherente a los datos, no se puede eliminar con el modelo

La **complejidad óptima** minimiza el error total de generalización.

In [ ]:
# ── Descomposición sesgo-varianza — regresión polinomial ──────────────────────
# Función verdadera: señal EEG simplificada (suma de sinusoides + ruido)
# Datos: muestras de un único ciclo

def f_true(x):
    """Función verdadera desconocida para el modelo."""
    return np.sin(2 * np.pi * x) + 0.5 * np.sin(4 * np.pi * x)

sigma_ruido = 0.4   # ruido irreducible
N_datasets  = 50    # conjuntos de entrenamiento para estimar la varianza
N_train     = 20    # puntos por conjunto de entrenamiento
x_test      = np.linspace(0, 1, 200)
y_test_true = f_true(x_test)

grados = [1, 3, 6, 12]   # grado del polinomio

fig, axes = plt.subplots(2, 4, figsize=(15, 7))

sesgo2_vals, var_vals, error_vals = [], [], []

for col, grado in enumerate(grados):
    predicciones = []

    for _ in range(N_datasets):
        x_tr = rng.uniform(0, 1, N_train)
        y_tr = f_true(x_tr) + rng.normal(0, sigma_ruido, N_train)
        coef = np.polyfit(x_tr, y_tr, grado)
        predicciones.append(np.polyval(coef, x_test))

    predicciones = np.array(predicciones)   # (N_datasets, 200)
    media_pred   = predicciones.mean(axis=0)

    sesgo2  = np.mean((media_pred - y_test_true) ** 2)
    var_mod = np.mean(predicciones.var(axis=0))
    error   = sesgo2 + var_mod + sigma_ruido**2

    sesgo2_vals.append(sesgo2)
    var_vals.append(var_mod)
    error_vals.append(error)

    # Fila superior: curvas de predicción
    ax = axes[0, col]
    for pred in predicciones[:15]:
        ax.plot(x_test, pred, alpha=0.12, color='steelblue', lw=1)
    ax.plot(x_test, y_test_true, 'k-', lw=2, label='f real')
    ax.plot(x_test, media_pred,  'tomato', lw=2, ls='--', label='Media pred.')
    ax.set(title=f'Grado {grado}', ylim=(-3, 3),
           xlabel='x' if col == 0 else '')
    if col == 0:
        ax.set_ylabel('y')
        ax.legend(fontsize=8)

# Fila inferior: descomposición sesgo-varianza
for col, grado in enumerate(grados):
    ax = axes[1, col]
    componentes = [sesgo2_vals[col], var_vals[col], sigma_ruido**2]
    colores_comp = ['tomato', 'steelblue', 'gray']
    etiquetas_c  = [f'Sesgo²={sesgo2_vals[col]:.3f}',
                     f'Varianza={var_vals[col]:.3f}',
                     f'Ruido={sigma_ruido**2:.3f}']
    ax.bar(etiquetas_c, componentes, color=colores_comp, alpha=0.8, edgecolor='white')
    ax.set_title(f'Error total = {error_vals[col]:.3f}', fontsize=9)
    ax.set_ylim(0, 0.9)
    ax.tick_params(axis='x', labelsize=8, labelrotation=15)

fig.suptitle(
    'Descomposición Sesgo-Varianza — regresión polinomial sobre señal EEG simplificada\n'
    'Cada curva azul = modelo entrenado en un conjunto distinto de 20 puntos',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

print('Resumen:')
print(f'{"Grado":>6}  {"Sesgo²":>8}  {"Varianza":>8}  {"Error total":>11}')
print('─' * 40)
for g, s2, v, e in zip(grados, sesgo2_vals, var_vals, error_vals):
    print(f'{g:>6}  {s2:>8.4f}  {v:>8.4f}  {e:>11.4f}')

## Parte 2 — Estrategias de validación cruzada para datos fisiológicos

La validación cruzada estima el error de generalización. Para datos biomédicos
la elección de la estrategia es crítica:

| Estrategia | Cuándo usar | Riesgo |
|---|---|---|
| **k-fold estándar** | Muestras i.i.d. (tabular, sin estructura temporal o de sujeto) | Fuga si hay estructura de sujeto |
| **k-fold estratificado** | Datos con desbalance de clases | Ídem |
| **LOSO** (Leave-One-Subject-Out) | Señales fisiológicas multi-sujeto (EEG, ECG, acelerometría) | Pesimista si hay pocos sujetos |
| **Ventana deslizante** | Series temporales | Fuga temporal si hay solapamiento |
| **CV anidada** | Selección de hiperparámetros + evaluación | Costosa pero honesta |

> **Regla fundamental:** la división train/test debe ocurrir **antes** de cualquier
> preprocesamiento que use información de los datos (normalización, selección de
> características, imputación). Cualquier paso que "vea" el conjunto de prueba introduce
> **fuga de información**.

In [ ]:
# ── Demostración de fuga de información: k-fold vs LOSO ───────────────────────
# Dataset: clasificación de imaginería motora EEG
# 10 sujetos, 40 ensayos por sujeto, 20 características
# Fuente del diseño experimental:
# Blankertz, B. et al. (2007). The non-invasive Berlin Brain-Computer Interface.
# IEEE Trans. Biomed. Eng., 54(12), 2141–2150.
# https://doi.org/10.1109/TBME.2007.906956

N_sujetos    = 10
N_ensayos    = 40
N_caracteris = 20
N_total      = N_sujetos * N_ensayos

# Simular datos: cada sujeto tiene su propia "firma" fisiológica
# La clase es genuinamente predecible (efecto real moderado)
# pero hay también un fuerte efecto de sujeto

sujeto_ids = np.repeat(np.arange(N_sujetos), N_ensayos)
etiquetas  = np.tile([0,1], N_total // 2)

# Efecto de sujeto (dominante) + efecto de clase (más débil)
offset_sujeto = rng.normal(0, 2.0, (N_sujetos, N_caracteris))  # firma fisiológica
efecto_clase  = rng.normal(0, 0.5, (2, N_caracteris))           # señal de clase

X = np.zeros((N_total, N_caracteris))
for i in range(N_total):
    s = sujeto_ids[i]
    c = int(etiquetas[i])
    X[i] = offset_sujeto[s] + efecto_clase[c] + rng.normal(0, 0.8, N_caracteris)

# ── Clasificador simple: distancia al centroide de clase ──────────────────────
def clasificar_centroide(X_tr, y_tr, X_te):
    """Clasificador de centroide mínimo (1-NN a los centroides de clase)."""
    c0 = X_tr[y_tr == 0].mean(axis=0)
    c1 = X_tr[y_tr == 1].mean(axis=0)
    d0 = np.linalg.norm(X_te - c0, axis=1)
    d1 = np.linalg.norm(X_te - c1, axis=1)
    return (d1 < d0).astype(int)

# ── k-fold estándar (INCORRECTO para datos multi-sujeto) ──────────────────────
K = 5
fold_size = N_total // K
idx_shuffled = rng.permutation(N_total)
accs_kfold = []
for k in range(K):
    idx_te = idx_shuffled[k*fold_size:(k+1)*fold_size]
    idx_tr = np.concatenate([idx_shuffled[:k*fold_size],
                               idx_shuffled[(k+1)*fold_size:]])
    # Normalizar DESPUÉS de dividir (correcto)
    mu_tr  = X[idx_tr].mean(axis=0)
    std_tr = X[idx_tr].std(axis=0) + 1e-8
    X_tr_n = (X[idx_tr] - mu_tr) / std_tr
    X_te_n = (X[idx_te] - mu_tr) / std_tr   # usa stats del train
    pred   = clasificar_centroide(X_tr_n, etiquetas[idx_tr], X_te_n)
    accs_kfold.append((pred == etiquetas[idx_te]).mean())

# ── LOSO (CORRECTO para datos multi-sujeto) ───────────────────────────────────
accs_loso = []
for s_te in range(N_sujetos):
    mask_te = sujeto_ids == s_te
    mask_tr = ~mask_te
    mu_tr   = X[mask_tr].mean(axis=0)
    std_tr  = X[mask_tr].std(axis=0) + 1e-8
    X_tr_n  = (X[mask_tr] - mu_tr) / std_tr
    X_te_n  = (X[mask_te] - mu_tr) / std_tr
    pred    = clasificar_centroide(X_tr_n, etiquetas[mask_tr], X_te_n)
    accs_loso.append((pred == etiquetas[mask_te]).mean())

# ── k-fold con fuga: normalizar ANTES de dividir ──────────────────────────────
mu_global  = X.mean(axis=0)
std_global = X.std(axis=0) + 1e-8
X_leaky    = (X - mu_global) / std_global   # ← fuga: usa info del test

accs_leaky = []
for k in range(K):
    idx_te = idx_shuffled[k*fold_size:(k+1)*fold_size]
    idx_tr = np.concatenate([idx_shuffled[:k*fold_size],
                               idx_shuffled[(k+1)*fold_size:]])
    pred = clasificar_centroide(X_leaky[idx_tr], etiquetas[idx_tr], X_leaky[idx_te])
    accs_leaky.append((pred == etiquetas[idx_te]).mean())

print('Comparación de estrategias de validación cruzada')
print('─' * 55)
print(f'k-fold estándar (k=5)     : {np.mean(accs_kfold):.3f} ± {np.std(accs_kfold):.3f}')
print(f'k-fold con FUGA           : {np.mean(accs_leaky):.3f} ± {np.std(accs_leaky):.3f}  ← inflado')
print(f'LOSO (correcto)           : {np.mean(accs_loso):.3f} ± {np.std(accs_loso):.3f}')
print()
print('El k-fold mezcla ensayos del mismo sujeto entre train y test.')
print('El modelo aprende la firma fisiológica del sujeto (no la clase).')
print('LOSO evalúa la generalización a sujetos NUEVOS — el único escenario real.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot de exactitudes
datos_box  = [accs_kfold, accs_leaky, accs_loso]
labels_box = ['k-fold\n(5 folds)', 'k-fold\ncon fuga', 'LOSO\n(correcto)']
colores_box = ['steelblue', 'tomato', 'seagreen']
bp = axes[0].boxplot(datos_box, patch_artist=True, widths=0.45)
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].axhline(0.5, color='gray', ls='--', lw=1, label='Azar')
axes[0].set(xticklabels=labels_box, ylabel='Exactitud',
            title='Validación cruzada — EEG imaginería motora\n'
                  'LOSO revela el rendimiento real de generalización')
axes[0].legend(fontsize=9)

# LOSO por sujeto
axes[1].bar(range(N_sujetos), accs_loso, color='seagreen', alpha=0.8, edgecolor='white')
axes[1].axhline(np.mean(accs_loso), color='navy', ls='--', lw=2,
                 label=f'Media={np.mean(accs_loso):.3f}')
axes[1].axhline(0.5, color='gray', ls=':', lw=1, label='Azar')
axes[1].set(xlabel='Sujeto', ylabel='Exactitud',
            title='Exactitud LOSO por sujeto\n'
                  'Alta variabilidad entre sujetos — típico en BCI')
axes[1].set_xticks(range(N_sujetos))
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## Parte 3 — Fuga de información: cuantificación del impacto

La fuga de información (*data leakage*) ocurre cuando el preprocesamiento usa información
del conjunto de prueba, introduciendo optimismo artificial en las métricas.

Fuentes comunes en señales biomédicas:

| Tipo de fuga | Ejemplo |
|---|---|
| **Normalización global** | Escalar con media/std de todo el dataset antes de dividir |
| **Selección de características** | Filtrar características correlacionadas con la etiqueta antes de CV |
| **Sujeto en train y test** | k-fold sin respetar la estructura de sujeto |
| **Ventanas solapadas** | Segmentos de la misma grabación en train y test |
| **Imputación global** | Imputar valores faltantes con estadísticas del conjunto completo |

In [ ]:
# ── Cuantificación del impacto de la fuga en selección de características ─────
# Escenario: selección por correlación con la etiqueta ANTES de la CV
# (error clásico en neuroimagen y EEG)
# Referencia: Kriegeskorte, N. et al. (2009). Circular analysis in systems
# neuroscience. Nature Neuroscience, 12(5), 535–540.
# https://doi.org/10.1038/nn.2303

N_vox  = 500   # vóxeles / canales (p. ej. fMRI o EEG)
N_samp = 60    # muestras (ensayos)
k_cv   = 5
n_top  = 20    # características a seleccionar

# Dataset puramente ruido — no hay señal real
X_null = rng.normal(0, 1, (N_samp, N_vox))
y_null = rng.integers(0, 2, N_samp).astype(float)

def exactitud_cv(X_data, y_data, k=5, seleccionar_antes=False, n_top=20):
    """
    Evalúa exactitud con k-fold.
    seleccionar_antes=True  → selección global antes de CV (fuga)
    seleccionar_antes=False → selección dentro de cada fold (correcto)
    """
    N = len(y_data)
    idx_all = rng.permutation(N)
    fold_sz = N // k
    accs = []

    # Fuga: selección global antes del loop
    if seleccionar_antes:
        corr_global = np.array([np.abs(np.corrcoef(X_data[:,j], y_data)[0,1])
                                  for j in range(X_data.shape[1])])
        top_idx_global = np.argsort(corr_global)[-n_top:]

    for fold in range(k):
        idx_te = idx_all[fold*fold_sz:(fold+1)*fold_sz]
        idx_tr = np.concatenate([idx_all[:fold*fold_sz],
                                   idx_all[(fold+1)*fold_sz:]])
        X_tr, X_te = X_data[idx_tr], X_data[idx_te]
        y_tr, y_te = y_data[idx_tr], y_data[idx_te]

        # Selección dentro del fold (correcto)
        if not seleccionar_antes:
            corr_tr = np.array([np.abs(np.corrcoef(X_tr[:,j], y_tr)[0,1])
                                  for j in range(X_tr.shape[1])])
            top_idx = np.argsort(corr_tr)[-n_top:]
        else:
            top_idx = top_idx_global

        X_tr_s = X_tr[:, top_idx]
        X_te_s = X_te[:, top_idx]

        # Normalización dentro del fold
        mu_tr  = X_tr_s.mean(axis=0)
        std_tr = X_tr_s.std(axis=0) + 1e-8
        X_tr_n = (X_tr_s - mu_tr) / std_tr
        X_te_n = (X_te_s - mu_tr) / std_tr

        # Clasificador: umbral en la media del score de correlación
        score_tr = X_tr_n @ (X_tr_n[y_tr==1].mean(0) - X_tr_n[y_tr==0].mean(0))
        umbral   = score_tr.mean()
        score_te = X_te_n @ (X_tr_n[y_tr==1].mean(0) - X_tr_n[y_tr==0].mean(0))
        pred     = (score_te > umbral).astype(int)
        accs.append((pred == y_te.astype(int)).mean())

    return np.array(accs)


# Repetir 100 veces para estimar la distribución del error
n_rep = 100
acc_correcto = []
acc_fuga     = []

for _ in range(n_rep):
    Xn = rng.normal(0, 1, (N_samp, N_vox))
    yn = rng.integers(0, 2, N_samp).astype(float)
    acc_correcto.append(exactitud_cv(Xn, yn, seleccionar_antes=False, n_top=n_top).mean())
    acc_fuga.append(    exactitud_cv(Xn, yn, seleccionar_antes=True,  n_top=n_top).mean())

acc_correcto = np.array(acc_correcto)
acc_fuga     = np.array(acc_fuga)

print('Impacto de la fuga en selección de características (datos NULOS — sin señal real):')
print(f'  Selección CORRECTA  (dentro del fold): {acc_correcto.mean():.3f} ± {acc_correcto.std():.3f}')
print(f'  Selección con FUGA  (antes de CV):     {acc_fuga.mean():.3f} ± {acc_fuga.std():.3f}')
print(f'  Inflación artificial: +{(acc_fuga.mean()-acc_correcto.mean())*100:.1f} pp')
print()
print('Con datos puramente de ruido, la exactitud correcta debe ser ~50%.')
print('La fuga infla artificialmente hasta ~70–80% — un resultado completamente falso.')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(acc_correcto, bins=20, alpha=0.6, color='seagreen',
         density=True, label=f'Correcto  μ={acc_correcto.mean():.2f}')
ax.hist(acc_fuga,     bins=20, alpha=0.6, color='tomato',
         density=True, label=f'Con fuga  μ={acc_fuga.mean():.2f}')
ax.axvline(0.5, color='k', ls='--', lw=1.5, label='Azar (0.50)')
ax.set(xlabel='Exactitud CV', ylabel='Densidad',
       title='Fuga en selección de características — datos sin señal real\n'
              'La fuga crea una ilusión de predectibilidad donde no existe')
ax.legend()
plt.tight_layout()
plt.show()

## Parte 4 — Comparaciones múltiples: Bonferroni y FDR

Cuando se realizan $m$ pruebas de hipótesis simultáneas, la probabilidad de obtener
al menos un falso positivo por azar es:

$$P(\text{al menos un FP}) = 1 - (1-\alpha)^m \approx m\alpha \quad (\text{para } m\alpha \text{ pequeño})$$

Con $m=100$ pruebas y $\alpha=0.05$: $P(\text{al menos un FP}) \approx 99.4\%$.

**Correcciones:**
- **Bonferroni:** $\alpha_\text{corr} = \alpha / m$ — controla la tasa de error por familia (FWER)
- **FDR (Benjamini-Hochberg 1995):** controla la fracción esperada de falsos positivos entre los rechazos — menos conservador, mayor potencia

In [ ]:
# ── Comparaciones múltiples en neuroimagen ────────────────────────────────────
# Simulamos un análisis de activación cerebral: 10,000 vóxeles
# Solo 200 vóxeles tienen activación real (señal verdadera)
# Referencia: Eklund, A. et al. (2016). Cluster failure: Why fMRI inferences for
# spatial extent have inflated false-positive rates. PNAS, 113(28), 7900–7905.
# https://doi.org/10.1073/pnas.1602413113

m_total   = 10_000   # total de pruebas (vóxeles)
m_reales  = 200      # vóxeles con activación real
m_nulos   = m_total - m_reales
alpha     = 0.05
N_grup    = 20       # sujetos por grupo

# Generar p-valores simulados
# Vóxeles nulos: p-valores uniformes
p_nulos  = rng.uniform(0, 1, m_nulos)
# Vóxeles reales: p-valores pequeños (efecto de tamaño d=0.6)
# Distribución no central: F(t) con nc = d * sqrt(N/2)
nc      = 0.6 * np.sqrt(N_grup / 2)
t_reales = rng.normal(nc, 1, m_reales)
p_reales = 2 * stats.norm.sf(np.abs(t_reales))   # bilateral

p_todos = np.concatenate([p_nulos, p_reales])
es_real = np.concatenate([np.zeros(m_nulos), np.ones(m_reales)]).astype(bool)

def aplicar_correcciones(p_vals, alpha=0.05):
    m = len(p_vals)
    rechazo_sin      = p_vals < alpha
    rechazo_bonf     = p_vals < alpha / m
    # FDR Benjamini-Hochberg
    orden   = np.argsort(p_vals)
    p_ord   = p_vals[orden]
    umbral_bh = alpha * np.arange(1, m+1) / m
    k_max   = np.where(p_ord <= umbral_bh)[0]
    rechazo_bh = np.zeros(m, dtype=bool)
    if len(k_max) > 0:
        rechazo_bh[orden[:k_max[-1]+1]] = True
    return rechazo_sin, rechazo_bonf, rechazo_bh

sin_corr, bonf, bh = aplicar_correcciones(p_todos, alpha)

def metricas_decision(rechazo, es_real):
    VP = (rechazo & es_real).sum()
    FP = (rechazo & ~es_real).sum()
    FN = (~rechazo & es_real).sum()
    VN = (~rechazo & ~es_real).sum()
    fdr_obs = FP / (VP + FP) if (VP + FP) > 0 else 0
    potencia = VP / es_real.sum()
    return VP, FP, FN, fdr_obs, potencia

print(f'Análisis de {m_total:,} vóxeles ({m_reales} con señal real)')
print(f'α = {alpha}\n')
print(f'{"Método":<22}  {"VP":>5}  {"FP":>6}  {"FDR_obs":>8}  {"Potencia":>9}')
print('─' * 58)
for nombre, rechazo in [('Sin corrección', sin_corr),
                          ('Bonferroni',     bonf),
                          ('FDR (B-H)',       bh)]:
    VP, FP, FN, fdr_obs, pot = metricas_decision(rechazo, es_real)
    print(f'{nombre:<22}  {VP:>5}  {FP:>6}  {fdr_obs:>8.1%}  {pot:>9.1%}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (nombre, rechazo) in zip(axes,
    [('Sin corrección', sin_corr), ('Bonferroni', bonf), ('FDR (B-H)', bh)]):

    VP, FP, FN, fdr_obs, pot = metricas_decision(rechazo, es_real)
    categorias = ['VP\n(detectados)', 'FP\n(falsos +)', 'FN\n(perdidos)']
    valores    = [VP, FP, FN]
    colores_b  = ['#16A34A', '#DC2626', '#F59E0B']
    ax.bar(categorias, valores, color=colores_b, alpha=0.8, edgecolor='white')
    ax.set_title(f'{nombre}\nFDR={fdr_obs:.1%}  Potencia={pot:.1%}', fontsize=10)
    for i, v in enumerate(valores):
        ax.text(i, v+20, str(v), ha='center', fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(valores) * 1.3)

plt.suptitle('Comparaciones múltiples — 10,000 vóxeles fMRI simulados',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Parte 5 — Prueba de permutaciones

La prueba de permutaciones es un método no paramétrico para evaluar la significancia
estadística sin asumir ninguna distribución de los datos:

1. Calcular el estadístico observado $T_\text{obs}$ (p. ej. AUROC, exactitud)
2. Permutar las etiquetas $B$ veces, recalculando $T_b$ en cada permutación
3. $p\text{-valor} = \frac{|\{b : T_b \geq T_\text{obs}\}|}{B}$

**Ventaja:** válida para cualquier estadístico, cualquier distribución, y cualquier
estructura de los datos. Es el estándar para validar modelos de BCI.

In [ ]:
# ── Prueba de permutaciones para el AUROC de un clasificador EEG ──────────────

# Usamos los datos de imaginería motora de la Parte 2
# Evaluamos el AUROC con LOSO vs la distribución nula por permutaciones

def auroc_loso(X_data, y_data, sujeto_ids):
    """Calcula el AUROC promedio con LOSO."""
    sujetos_unicos = np.unique(sujeto_ids)
    aurocs = []
    for s_te in sujetos_unicos:
        mask_te = sujeto_ids == s_te
        mask_tr = ~mask_te
        mu_tr   = X_data[mask_tr].mean(axis=0)
        std_tr  = X_data[mask_tr].std(axis=0) + 1e-8
        X_tr_n  = (X_data[mask_tr] - mu_tr) / std_tr
        X_te_n  = (X_data[mask_te] - mu_tr) / std_tr
        c1_mean = X_tr_n[y_data[mask_tr] == 1].mean(axis=0)
        c0_mean = X_tr_n[y_data[mask_tr] == 0].mean(axis=0)
        scores  = X_te_n @ (c1_mean - c0_mean)
        y_te    = y_data[mask_te]
        if len(np.unique(y_te)) < 2:
            continue
        # AUROC manual
        pos_scores = scores[y_te == 1]
        neg_scores = scores[y_te == 0]
        auroc_s = np.mean([np.mean(p > neg_scores) for p in pos_scores])
        aurocs.append(auroc_s)
    return np.mean(aurocs)


auroc_obs = auroc_loso(X, etiquetas, sujeto_ids)

# Distribución nula: permutar etiquetas manteniendo la estructura de sujeto
n_perm = 500
aurocs_perm = []
etiquetas_perm = etiquetas.copy()

for _ in range(n_perm):
    # Permutar etiquetas dentro de cada sujeto (permutación restringida)
    etiq_perm = etiquetas.copy()
    for s in range(N_sujetos):
        mask_s = sujeto_ids == s
        etiq_perm[mask_s] = rng.permutation(etiquetas[mask_s])
    aurocs_perm.append(auroc_loso(X, etiq_perm, sujeto_ids))

aurocs_perm = np.array(aurocs_perm)
p_perm = np.mean(aurocs_perm >= auroc_obs)

print(f'AUROC observado (LOSO): {auroc_obs:.4f}')
print(f'Distribución nula:      μ={aurocs_perm.mean():.4f}  σ={aurocs_perm.std():.4f}')
print(f'p-valor (permutaciones): {p_perm:.4f}')
print()
if p_perm < 0.05:
    print('→ El clasificador es significativamente mejor que el azar (p<0.05)')
else:
    print('→ No hay evidencia de que el clasificador supere el azar (p≥0.05)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(aurocs_perm, bins=30, density=True, color='steelblue',
         alpha=0.7, edgecolor='white', label='Distribución nula (permutaciones)')
ax.axvline(auroc_obs, color='tomato', lw=2.5,
            label=f'AUROC observado = {auroc_obs:.3f}')
pct_extremo = np.percentile(aurocs_perm, 95)
ax.axvline(pct_extremo, color='gray', ls='--', lw=1.5,
            label=f'Percentil 95% = {pct_extremo:.3f}')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1]>0 else 10],
                  pct_extremo, max(aurocs_perm.max(), auroc_obs)+0.05,
                  alpha=0.15, color='tomato', label=f'p = {p_perm:.3f}')
ax.set(xlabel='AUROC', ylabel='Densidad',
       title=f'Prueba de permutaciones — clasificador BCI (N={N_sujetos} sujetos, LOSO)\n'
              'La distribución nula se construye permutando etiquetas')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## ✏️ Ejercicios

1. **Curvas de aprendizaje.** Implementa curvas de aprendizaje para el clasificador
   de la Parte 2: entrena con subconjuntos crecientes del set de entrenamiento
   (10%, 20%, …, 100%) y grafica el error de train y validación vs el tamaño.
   ¿El modelo sufre de sesgo alto, varianza alta, o está bien equilibrado?

2. **CV anidada.** Implementa validación cruzada anidada (outer loop: LOSO;
   inner loop: k-fold para seleccionar el número de características).
   Compara el AUROC estimado con el de la CV simple. ¿Cuánto optimismo introduce
   la selección de hiperparámetros sin CV anidada?

3. **FDR en EEG.** Simula un análisis de conectividad EEG: 64 electrodos × 64 electrodos
   = 2016 pares únicos. Solo 50 pares tienen coherencia real aumentada.
   Aplica las tres correcciones (sin corrección, Bonferroni, B-H) y compara VP, FP y
   potencia. ¿Cuántos pares reales se pierden con Bonferroni?

4. **Prueba de permutaciones vs t-test.** Para el escenario de la Parte 5, compara
   el p-valor de la prueba de permutaciones con el de una prueba t de una muestra
   (H₀: AUROC = 0.5). ¿Cuándo difieren? ¿Cuál es más conservador?

5. *(Desafío)* **Fuga temporal en series de tiempo.** Genera una serie temporal
   de 1000 puntos con autocorrelación (AR(1) con φ=0.9). Divide en train/test
   (a) aleatoriamente y (b) temporalmente (train=primeros 700, test=últimos 300).
   Ajusta un modelo autorregresivo y compara el RMSE. ¿Cuánto infla la división
   aleatoria en presencia de autocorrelación?

## 📚 Conjuntos de datos utilizados / referenciados

| Conjunto de datos | Fuente | Notas |
|---|---|
| BCI Competition IV 2a (diseño) | Blankertz, B. et al. (2007). *IEEE TBME* 54(12):2141–2150. https://doi.org/10.1109/TBME.2007.906956 | Imaginería motora — datos simulados calibrados con este diseño |
| Análisis fMRI (diseño) | Eklund, A. et al. (2016). *PNAS* 113(28):7900–7905. https://doi.org/10.1073/pnas.1602413113 | Comparaciones múltiples en neuroimagen |
| PhysioNet EEG Motor Movement | https://physionet.org/content/eegmmidb/ | 109 sujetos, imaginería motora y ejecución |